## Registering FTIR Masks with H&E Images

**Aim:** register and align the masks from FTIR images of human breast tissue biopsy cores (malignant and non-malignant) with their corresponding H&E images. 

In [ ]:
import cv2
import numpy as np

def process_mask(ftir_image_path, he_image_path, he_mask_path):
    ftir_img = cv2.imread(ftir_image_path, cv2.IMREAD_COLOR)
    he_img = cv2.imread(he_image_path)
    he_height, he_width = he_img.shape[:2]

    resized_ftir = cv2.resize(ftir_img, (he_width, he_height), interpolation=cv2.INTER_NEAREST)

    # Annotation colours (BGR Format)
    annotations = [
        (0, 0, 255),       # Red: malignant epithelium
        (156, 55, 165),    # Purple: malignant stroma
        (66, 193, 103),    # Green: non-malignant epithelium
        (53, 143, 243)     # Orange: non-maligant stroma
    ]

    processed_mask = np.zeros_like(resized_ftir)
    for color in annotations:
        mask = np.all(resized_ftir == color, axis=-1)
        processed_mask[mask] = color

    cv2.imwrite(he_mask_path, processed_mask)

# process_mask('dataset/FTIR_core_train_masks/B9.png', 'dataset/HE_core_train_images/B9.tif', 'dataset/HE_core_train_masks/B9.png')


In [ ]:
def overlay_annotations(he_image_path, processed_mask_path, output_path):
    he_img = cv2.imread(he_image_path)
    he_img = cv2.cvtColor(he_img, cv2.COLOR_BGR2RGB)

    mask_img = cv2.imread(processed_mask_path)
    mask_img = cv2.cvtColor(mask_img, cv2.COLOR_BGR2RGB)

    output_img = he_img.copy()

    # Annotation colours (RGB format)
    colors = {
        'red': [255, 0, 0],
        'purple': [165, 55, 156],
        'green': [103, 193, 66],
        'orange': [243, 143, 53]
    }

    for color_name, color_value in colors.items():
        mask = np.all(mask_img == color_value, axis=-1)
        if np.any(mask):
            output_img[mask] = color_value

    cv2.imwrite(output_path, cv2.cvtColor(output_img, cv2.COLOR_RGB2BGR))

# overlay_annotations('dataset/HE_core_train_images/B9.tif', 'dataset/HE_core_train_masks/B9.png', 'dataset/HE_train_masks_overlay/B9.jpg')


In [ ]:
import os

def process_dataset(ftir_mask_dir, he_image_dir, he_mask_dir, he_overlay_dir):
    ftir_masks = [f for f in os.listdir(ftir_mask_dir) if f.endswith('.png')]

    for f in ftir_masks:
        core = f.split('.')[0]
        
        he_image_path = os.path.join(he_image_dir, core + '.tif')
        he_mask_path = os.path.join(he_mask_dir, core + '.png')
        he_overlay_path = os.path.join(he_overlay_dir, core + '.png')
        ftir_image_path = os.path.join(ftir_mask_dir, f)
        
        process_mask(ftir_image_path, he_image_path, he_mask_path)
        overlay_annotations(he_image_path, he_mask_path, he_overlay_path)

# Training set
train_ftir_mask_dir = 'dataset/FTIR_core_train_masks'
train_he_image_dir = 'dataset/HE_core_train_images'
train_he_mask_dir = 'dataset/HE_core_train_masks'
train_he_overlay_dir = 'dataset/HE_train_masks_overlay'

process_dataset(train_ftir_mask_dir, train_he_image_dir, train_he_mask_dir, train_he_overlay_dir)

# Independent test set
test_ftir_mask_dir = 'dataset/FTIR_core_test_masks'
test_he_image_dir = 'dataset/HE_core_test_images'
test_he_mask_dir = 'dataset/HE_core_test_masks'
test_he_overlay_dir = 'dataset/HE_test_masks_overlay'

process_dataset(test_ftir_mask_dir, test_he_image_dir, test_he_mask_dir, test_he_overlay_dir)
